# Week 5 Exercise — Bugs RAG & Gradio Chat

This notebook builds a **RAG-backed chat app** so we can talk to our bug dataset in natural language (when/where/how bugs were injected, severity, tags, code context, etc.).

**Where we are**
- **Eval** is in place: `tests.jsonl`, `test.py`, and `eval_bugs.py` run retrieval + answer evaluation (OpenRouter judge). Implementation stubs live in `implementation/answer.py`.
- **Data**: `bugs/*.md` (one file per bug with correct/buggy code, types, lines, tags, timestamps) and `buggy_dataset.json` / `buggy_dataset_nl.jsonl` from the Week 3 Synthetic Buggy Code Factory.

**Plan**
1. **Ingest** — **LangChain + manual RAG hybrid**: use LangChain for loaders and Chroma (day2-style), but **no Hugging Face**. We use **OpenRouter** for embeddings (three candidate models). An **LLM-as-expert** will inspect our dataset/structure and recommend the best **text-splitting** config (chunk size, overlap, etc.); we then create the vector store in our specified DB similarly to day2.
2. **Answer** — Implement `fetch_context` and `answer_question` in `implementation/answer.py` using that vector store and an LLM (OpenRouter).
3. **Chat UI** — **Gradio** chat app to query the bugs knowledge base.

Below: installs, imports, and env verification.

**Dependencies:** If you see `No module named pip`, the notebook kernel (e.g. `.venv` from uv) may not have pip. Install from the repo root instead: `uv sync` or `uv pip install python-dotenv langchain langchain-chroma chromadb langchain-openai langchain-community langchain-text-splitters langchain-core tiktoken numpy scikit-learn plotly pydantic litellm tqdm gradio`. Then restart the kernel. If your env already has these, skip the cell below.

In [ ]:
%pip install -q python-dotenv langchain langchain-chroma chromadb langchain-openai langchain-community langchain-text-splitters langchain-core tiktoken numpy scikit-learn plotly pydantic litellm tqdm gradio

In [ ]:
import os
import glob
from pathlib import Path

import numpy as np
import tiktoken
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from sklearn.manifold import TSNE
import plotly.graph_objects as go

from pydantic import BaseModel, Field
from litellm import completion
from tqdm import tqdm
import gradio as gr

In [ ]:
MODEL = "openrouter/openai/gpt-4o-mini"
db_name = "vector_db_bugs"
load_dotenv(override=True)

THIS_DIR = Path.cwd()
BUGS_PATH = THIS_DIR / "bugs"
DB_PATH = THIS_DIR / db_name

EMBEDDING_OPTIONS = [
    "text-embedding-3-large",
    "text-embedding-3-small",
    "text-embedding-ada-002",
]

openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
files = sorted(BUGS_PATH.glob("*.md"))
doc_count = len(files)
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

total_chars = 0
total_tokens = 0
for f in files:
    text = f.read_text(encoding="utf-8")
    total_chars += len(text)
    total_tokens += len(encoding.encode(text))

### Ingest: Load documents → LLM-expert (splitter config) → Chunk → Embed → Chroma

Load the bugs corpus with LangChain, ask an LLM to recommend chunk size and overlap from the dataset stats, split into chunks, then embed and persist to Chroma (day2-style).

In [ ]:
from implementation.ingest import load_documents

documents = load_documents(BUGS_PATH)

In [ ]:
from implementation.ingest import get_splitter_config

splitter_type, chunk_size, chunk_overlap = get_splitter_config(
    documents, total_chars, total_tokens, MODEL, openai_api_key
)

In [ ]:
from implementation.ingest import create_chunks

chunks = create_chunks(documents, splitter_type, chunk_size, chunk_overlap)

In [ ]:
from implementation.ingest import create_vectorstore

OPENROUTER_BASE = "https://openrouter.ai/api/v1"
embedding_model = EMBEDDING_OPTIONS[0]
vectorstore = create_vectorstore(chunks, DB_PATH, embedding_model, OPENROUTER_BASE)
collection = vectorstore._collection

### Evaluation dashboard (Gradio)

Run retrieval and answer evaluation visually. Uses `evals` (tests, metrics, LLM judge) and `implementation.answer`. Launch the app below.

In [ ]:
import sys

if str(THIS_DIR) not in sys.path:
    sys.path.insert(0, str(THIS_DIR))

from evals.evaluator import launch

launch(inbrowser=True)

In [ ]:
# Evaluation runs in the Gradio app launched above (retrieval + answer tabs).

In [ ]:
import gradio as gr
from dotenv import load_dotenv

from implementation.answer import answer_question

load_dotenv(override=True)

def format_context(context):
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata.get('source', 'unknown')}</span>\n\n"
        result += doc.page_content + "\n\n"
    return result

def chat(history):
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = answer_question(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(context)

def put_message_in_chatbot(message, history):
    return "", history + [{"role": "user", "content": message}]

theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

# Added standard inputs and outputs logic for gradio chatbots
with gr.Blocks(title="Bugs Knowledge Base Assistant", theme=theme) as ui:
    gr.Markdown("# 🐛 Bugs Knowledge Base Assistant\nAsk me anything about the codebase bugs!")

    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="💬 Conversation", height=600, type="messages", show_copy_button=True
            )
            message = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about the bugs...",
                show_label=False,
            )

        with gr.Column(scale=1):
            context_markdown = gr.Markdown(
                label="📚 Retrieved Context",
                value="*Retrieved context will appear here*",
                container=True,
                height=600,
            )

    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(chat, inputs=chatbot, outputs=[chatbot, context_markdown])

ui.launch(inbrowser=True)